# Beam α + κ — fast probe prior

**Role:** before a full-sky metacatalog detect, estimate per-band size **α** and
shared zenith-south **κ** from a short Dec ladder of tiles. Writes
`beam_scale.json` for `ovro_lwa_healpix_tile_detect.ipynb`.

**No VLSSR.** Hybrid rescale is deferred to
`beam_alpha_kappa_estimate.ipynb` after `radio_crossmatch.ipynb` (validation).

**Maps:** prefer existing `OUTPUT_DIR/healpix_{band}_nside*.fits` (from the
detect notebook map cells). **Beam / RESTFRQ** come from the prebuilt mosaic
Primary (rewritten MAP+WEIGHT do not store `BMAJ`/`BMIN`).

Probe tiles: `DEFAULT_ALPHA_KAPPA_PROBE_TILES` = `(28, 102, 189)`. Detect with
`α=1`, `κ=0`, then size-only estimate (`apply_vlssr_hybrid=False`).


In [ ]:
from pathlib import Path

import pandas as pd
from lwa_healpix import read_healpix_fits, write_healpix_fits

from lwa_catalog.analyze import (
    DEFAULT_ALPHA_KAPPA_PROBE_TILES,
    BeamAlphaKappaEstimateConfig,
    estimate_beam_alpha_kappa,
    summarize_beam_alpha_kappa_estimate,
    write_beam_alpha_kappa_result,
)
from lwa_catalog.analyze.healpix_cutout import healpix_coadd_path
from lwa_catalog.create.healpix_detect import (
    DEFAULT_BEAM_SCALE,
    DEFAULT_DETECT_TIER_THRESHOLDS,
    detect_sources_on_healpix_tiles,
)
from lwa_catalog.create.healpix_prebuilt import (
    beam_and_restfreq_from_healpix_fits,
    discover_prebuilt_healpix_maps,
    list_mhz_subband_dirs,
    load_prebuilt_healpix_map,
)
from lwa_catalog.io import write_table
from lwa_catalog.paths import CatalogLayout

OUTPUT_DIR = Path("/fast/claw/metacatalog_healpix_IdTR-0.75s_subband5")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
layout = CatalogLayout(OUTPUT_DIR)

# Same prebuilt root as ovro_lwa_healpix_tile_detect.ipynb (beam keywords).
HEALPIX_ROOT = Path(
    "/lustre/pipeline/exopipe/phase3/mosaic/Run_FEATHER3_20241218-20251029"
)
HEALPIX_GLOB = (
    "??MHz/I/healpix/*_I_deep_Taper_Robust-0.75_coadd*"
    "mosaic_jybeam_feather_allsky*.fits"
)
HEALPIX_PREFER_SUBSTRING = "ssub-rst8"

NSIDE_MAP = 2048
NSIDE_TILE = 4
TILE_ALIGN = "diamond"
TILE_MARGIN = 0.05
OVERLAP = 0.2
COORD_FRAME = "icrs"
NESTED = True

PROBE_TILES = DEFAULT_ALPHA_KAPPA_PROBE_TILES
BANDS = None  # None → all *MHz dirs under HEALPIX_ROOT
WRITE_OUTPUT = True
REUSE_PROBE_CATALOGS = True
REUSE_HEALPIX_FITS = True

BDSF_KW = dict(
    thresh="hard",
    rms_map=True,
    savefits_rmsim=False,
    outdir=str(OUTPUT_DIR / "beam_scale_probe"),
    rms_box=(120, 30),
    adaptive_rms_box=True,
    rms_box_bright=(40, 10),
    mean_map="map",
    adaptive_thresh=150.0,
    atrous_do=False,
    psf_vary_do=False,
    quiet=True,
    ncores=1,
)
DETECT_TIER_THRESHOLDS = DEFAULT_DETECT_TIER_THRESHOLDS

PROBE_DIR = OUTPUT_DIR / "beam_scale_probe"
PROBE_DIR.mkdir(parents=True, exist_ok=True)


def probe_catalog_path(band: str) -> Path:
    return PROBE_DIR / f"sources_healpix_probe_tiles_{band}.parquet"


COLOR_BANDS = (
    tuple(BANDS)
    if BANDS is not None
    else tuple(list_mhz_subband_dirs(HEALPIX_ROOT))
)
prebuilt_paths = discover_prebuilt_healpix_maps(
    HEALPIX_ROOT,
    relative_glob=HEALPIX_GLOB,
    prefer_substring=HEALPIX_PREFER_SUBSTRING,
)
print(f"bands: {len(COLOR_BANDS)}  probe tiles: {PROBE_TILES}")
print(f"prebuilt maps found: {len(prebuilt_paths)}")


In [ ]:
frames: list[pd.DataFrame] = []
for band in COLOR_BANDS:
    out = probe_catalog_path(band)
    if REUSE_PROBE_CATALOGS and out.is_file():
        cat = pd.read_parquet(out)
        print(f"{band}: reused probe catalog rows={len(cat):,} ← {out.name}")
        frames.append(cat)
        continue

    src = prebuilt_paths.get(band)
    if src is None:
        print(f"{band}: skip (no prebuilt HEALPix)")
        continue

    healpix_fits = healpix_coadd_path(OUTPUT_DIR, band, nside=NSIDE_MAP)
    if REUSE_HEALPIX_FITS and healpix_fits.is_file():
        healpix_map, weight, meta = read_healpix_fits(healpix_fits)
        print(f"{band}: reused map {healpix_fits.name}")
    else:
        healpix_map, weight, meta = load_prebuilt_healpix_map(src)
        if int(meta["nside"]) != int(NSIDE_MAP):
            raise ValueError(
                f"{band}: prebuilt nside={meta['nside']} != NSIDE_MAP={NSIDE_MAP}"
            )
        write_healpix_fits(
            healpix_fits,
            healpix_map,
            weight,
            nside=NSIDE_MAP,
            nested=NESTED,
            coord_frame=COORD_FRAME,
            overwrite=True,
        )
        print(f"{band}: wrote {healpix_fits.name} from {src.name}")

    bmaj, bmin, bpa, restfreq = beam_and_restfreq_from_healpix_fits(src)
    print(
        f"  detect tiles {PROBE_TILES}  "
        f"header BMAJ/BMIN={bmaj:.5f}/{bmin:.5f} (α=1, κ=0)"
    )
    cat = detect_sources_on_healpix_tiles(
        healpix_map,
        weight,
        nside_map=NSIDE_MAP,
        nside_tile=NSIDE_TILE,
        align=TILE_ALIGN,
        margin=TILE_MARGIN,
        overlap=OVERLAP,
        coord_frame=COORD_FRAME,
        nested=NESTED,
        bmaj=bmaj,
        bmin=bmin,
        bpa=bpa,
        beam_scale=DEFAULT_BEAM_SCALE,
        restfreq_hz=restfreq,
        band=band,
        bdsf_kw=BDSF_KW,
        tier_thresholds=DETECT_TIER_THRESHOLDS,
        skip_empty=True,
        tile_ipix=PROBE_TILES,
    )
    write_table(cat, out)
    print(f"  wrote {out.name} rows={len(cat):,}")
    frames.append(cat)

sources = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
print(f"probe sources total: {len(sources):,}")
if len(sources) and "tile_ipix" in sources.columns:
    print(sources.groupby(["band", "tile_ipix"]).size().unstack(fill_value=0))


In [ ]:
config = BeamAlphaKappaEstimateConfig(
    beam_scale_used=1.0,
    apply_vlssr_hybrid=False,
    bands=None if BANDS is None else tuple(BANDS),
)
result = estimate_beam_alpha_kappa(sources, config=config, metacatalog_vlssr=None)
print(summarize_beam_alpha_kappa_estimate(result))
display(result.by_band)

if WRITE_OUTPUT:
    path = write_beam_alpha_kappa_result(
        result,
        layout,
        extra={
            "source": "alpha_kappa_probe_prior",
            "probe_tiles": list(PROBE_TILES),
        },
    )
    print("wrote", path.resolve())
    print("Next: ovro_lwa_healpix_tile_detect.ipynb loads this beam_scale.json")
    print(
        "After full detect + radio_crossmatch: beam_alpha_kappa_estimate.ipynb "
        "(validation / optional VLSSR hybrid)"
    )
